# MAPK Base-Editing Screens — Library Design

Design of the tiling ABE/CBE sgRNA library the screens were run with: 15,406 guides across 22 genes of the MAPK pathway. The guides counted in `TableS1-ScreenData.xlsx` are the guides this notebook produces.

| Notebook section | Pipeline step | Writes |
|---|---|---|
| 3 | 1 — MANE Select transcript per gene | `01_*` |
| 4 | 2 — ABE and CBE sgRNA designs | `02_*` |
| 5 | 3 — merge the two editors and filter | `03_*` |
| 6 | 4 — per-gene library statistics | `04_*` |
| 7 | 5 — BEhive predicted editing efficiency | `05_*` |
| 8 | 6 — FlashFry off-target specificity | `06_*` |
| 9 | 7 — the final library | `07_MAPK_lib_final.csv` |

Section 0 provisions the environment from scratch, so no pre-existing conda environment is needed. Sections 4 onward are independent of each other and can be run selectively once sections 0–3 have run.

Steps 2, 5 and 6 need resources the rest of the pipeline does not:
1.   A live Ensembl endpoint and a 3.9 GB ClinVar dump
2.   A pinned Python 3.7 environment
3.   A multi-gigabyte off-target database

Results for all three are in the workbook, so with `RUN_HEAVY_STEPS = False` (the default) the notebook reproduces the published library from cache.

## 0. Environment

Installs the pinned dependency set into the runtime. In Colab, also clones the
repository if the notebook is running standalone.

In [ ]:
#@title Install dependencies { display-mode: "form" }
# Provisions the runtime from scratch on every run, so no persistent environment is required. Takes a few minutes on a cold Colab runtime.

import os
import subprocess
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    # The repository holds the code and TableS6-LibraryDesign.xlsx.
    # Set REPO_URL to clone from git, or mount Drive and point REPO_DIR at the copy there.
    REPO_DIR = "/content" # @param {type:"string"}
    REPO_DIR = Path(REPO_DIR)
    REPO_URL = "" # @param {type:"string"}

    if REPO_URL and not REPO_DIR.exists():
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)], check=True)
    elif not REPO_DIR.exists():
        from google.colab import drive
        drive.mount("/content/drive")
        raise SystemExit(
            f"{REPO_DIR} not found. Set REPO_URL, or set REPO_DIR to the "
            "repository folder inside /content/drive."
        )
    os.chdir(REPO_DIR)

REPO_ROOT = Path.cwd()
while not (REPO_ROOT / "TableS1-ScreenData.xlsx").exists() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
os.chdir(REPO_ROOT)
print("Repository root:", REPO_ROOT)

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", f"{REPO_DIR}/requirements.txt"], check=True)

print("Environment ready.")

## 1. Imports

Every import used anywhere in the notebook.

In [ ]:
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

# Manuscript-specific code.
sys.path.insert(0, str(Path.cwd()))
# `code` is also a Python standard-library module.
sys.modules.pop("code", None)
from code import config as cfg, library_design as lib
print("code/ modules loaded.")

warnings.filterwarnings("ignore")
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 50)

## 2. Run settings

Input paths, base-editor names, and the Ensembl endpoint are defined once in code/config.py and used by every section below.

In [ ]:
#@title Run settings { display-mode: "form" }

# False: use the results in TableS6-LibraryDesign.xlsx for steps 2, 5 and 6.
# True:  recompute what this kernel and this machine can, and report what they cannot.
RUN_HEAVY_STEPS = False # @param {type:"boolean"}

# Reuse designs left by a previous step-2 run instead of designing again.
REUSE_EXISTING_DESIGNS = True # @param {type:"boolean"}

# Ensembl REST endpoint step 2 designs against. The default is the release-pinned
# archive for release 110, the release current when the published library was
# designed. See code/config.py for the alternatives and the one to avoid.
ENSEMBL_SERVER = "https://jul2023.rest.ensembl.org" # @param {type:"string"}

cfg.make_library_design_dirs()
OUT = cfg.OUT_LIBRARY_DESIGN

ABE_NAME, CBE_NAME = cfg.LIB_ABE_NAME, cfg.LIB_CBE_NAME

print(f"Design inputs:   {cfg.LIBRARY_DESIGN_XLSX.name}")
print(f"Output:          {OUT}")
print(f"Base editors:    {ABE_NAME} (A->G, NG PAM), {CBE_NAME} (C->T, NG PAM)")
print(f"RUN_HEAVY_STEPS: {RUN_HEAVY_STEPS}")
print(f"Kernel:          {sys.executable}")

## 3. Gene selection and transcript mapping

Each target gene is mapped to its MANE Select transcript, which is the input the design script expects. Restricting the MANE v1.0 summary to `MANE Select` yields exactly one transcript per gene; that join is done once by `code/build_inputs.py` and stored as the workbook's `Transcripts` sheet, so nothing here has to read the 19,120-row summary.

In [ ]:
transcripts, design_input_path = lib.map_genes_to_transcripts(OUT)

print(f"{len(transcripts)} genes mapped to MANE Select transcripts")
print(f"Design script input: {design_input_path.name}")
transcripts

## 4. sgRNA design

The design script (`base_editing_guide_designs.py`, Broad Institute; author Mudra Hegde) tiles every sgRNA whose editing window falls within a coding exon of the target transcript plus a 30 nt intron buffer and annotates the resulting amino-acid changes.

Editor parameters come from the workbook's `BaseEditors` sheet, built from `inputs/design_scripts/BEs_to_use.txt`. The designs used for the published library are the workbook's `SpG-ABE8e` and `SpG-BE3.9` sheets. Regenerating them needs ClinVar `variant_summary.txt` in `inputs/design_scripts/` and access to an Ensembl REST endpoint, and takes about an hour. The cell below checks both and reports if either is unavailable. The release used is recorded in `02_design_provenance.json`.

In [ ]:
#@title Base editor parameters { display-mode: "form" }

be_params = lib.load_be_parameters()
print(be_params.to_string(index=False))

In [ ]:
#@title Resolve the sgRNA designs { display-mode: "form" }

designs = lib.resolve_designs(
    transcripts, design_input_path, OUT,
    run_heavy=RUN_HEAVY_STEPS, reuse_existing=REUSE_EXISTING_DESIGNS,
    server=ENSEMBL_SERVER,
)

for note in designs["notes"]:
    print(f"  {note}")
if designs["blockers"]:
    print("\nDesigns were not regenerated:")
    for blocker in designs["blockers"]:
        print(f"  - {blocker}")

abe_designs, cbe_designs = designs["designs"][ABE_NAME], designs["designs"][CBE_NAME]

print(f"\nsource={designs['source']}  regenerated={designs['regenerated']}")
for editor, table in designs["designs"].items():
    print(f"  {editor}: {len(table):,} designed sgRNAs "
          f"from {designs['origins'][editor]}")

## 5. Merge and filter

Both editors are designed against the same NG PAM guide space, so most sgRNAs carry both an A to G and a C to T annotation. Guides are removed if they edit the start codon, create or remove a stop codon, edit only intronic sequence, edit only UTR sequence, or occur more than once in the library.

In [ ]:
master_BE_df, gene_list, unfiltered_path = lib.merge_designs(
    abe_designs, cbe_designs, OUT)

print(f"merged designs: {len(master_BE_df):,} guides across {len(gene_list)} genes")
print(f"wrote {unfiltered_path.name}, which step 5 scores in full")

In [ ]:
#@title Filter { display-mode: "form" }

filtered, filter_counts = lib.filter_library(master_BE_df, OUT)

print(filter_counts.to_string(index=False))
print(f"\n{len(master_BE_df):,} -> {len(filtered):,} guides retained")

In [ ]:
library, library_path = lib.number_guides(filtered, gene_list, OUT)

print(f"wrote {library_path.name}: {len(library):,} guides")
library.head()

## 6. Per-gene statistics

The number of unique missense mutations the library can install and the number of distinct residues affected for each gene for ABE and CBE separately.

In [ ]:
stats_df = lib.gene_statistics(library, gene_list, transcripts, OUT)

print(f"{len(stats_df)} genes, {stats_df['Total guides'].sum():,} guides")
stats_df

## 7. BEhive predicted editing efficiency

[BEhive](https://github.com/maxwshen/be_predict_efficiency) predicts base editing efficiency from the 50 bp of genomic context centred on the protospacer (20 nt upstream, the 20 nt guide, and 10 nt downstream). Scores are computed for both editors across all designed guides. `BE4` is used for the CBE and `ABE8` for the ABE with `mES` as the cell type.

Recomputing needs the pinned Python 3.7 environment built by `bash library_design_JW/inputs/behive/setup_behive_env.sh`, selected as the "Python (BEhive_env)" kernel. Its `scikit-learn==0.20.3` pin is a strict requirement. A recomputed run is cross-checked against the published scores.

In [ ]:
#@title Resolve the BEhive scores { display-mode: "form" }

behive = lib.resolve_behive_scores(unfiltered_path, transcripts, OUT,
                                   run_heavy=RUN_HEAVY_STEPS)

for note in behive["notes"]:
    print(f"  {note}")
if behive["blockers"]:
    print("\nBEhive was not recomputed:")
    for blocker in behive["blockers"]:
        print(f"  - {blocker}")
if behive["cross_check"] is not None:
    print("\nCross-check against the published scores:")
    print(behive["cross_check"].to_string(index=False))

behive_scores = behive["scores"]
print(f"\nsource={behive['source']}  "
      f"BEhive scores for {len(behive_scores):,} unique sgRNAs")
behive["scored"][["sgRNA sequence", "Gene Symbol", "context_50bp",
                  "BEhive_CBE", "BEhive_ABE"]].head()

## 8. FlashFry off-target specificity

FlashFry scores each guide against a genome-wide off-target database. It accepts only NGG guides and discards anything else, which would exclude most of this library, since SpG guides require only NG. A `G` is therefore appended to each guide's NG PAM to form a 23 nt sequence FlashFry will accept. Only the 20 nt protospacer determines the off-target search; the appended base is removed when scores are merged back.

In [ ]:
#@title Write the query FASTA { display-mode: "form" }

library_with_pam, flashfry_fasta = lib.write_flashfry_fasta(library_path, OUT)

print(f"wrote {flashfry_fasta.name}: {len(library_with_pam):,} sequences")
library_with_pam[["guide_id", "sgRNA sequence", "PAM", "sgRNA sequence w NGG"]].head()

In [ ]:
#@title Load the specificity scores { display-mode: "form" }

FLASHFRY_SCORED = None

specificity = lib.resolve_specificity_scores(
    OUT, FLASHFRY_SCORED, run_heavy=RUN_HEAVY_STEPS)

for note in specificity["notes"]:
    print(f"  {note}")
for blocker in specificity["blockers"]:
    print(f"  - {blocker}")

specificity_scores = specificity["scores"]
print(f"\nsource={specificity['source']}  "
      f"specificity scores for {len(specificity_scores):,} unique sgRNAs")
specificity["scored"][["target", "Hsu2013", "DoenchCFD_specificityscore",
                       "otCount"]].head()

## 9. Final library

*Pipeline step 7.* The filtered library joined to its BEhive efficiency predictions and FlashFry specificity scores.

In [ ]:
final_df, final_path = lib.assemble_library(
    library_path, behive_scores, specificity_scores, OUT)

print(f"wrote {final_path.name}")
print(f"  guides          {len(final_df):,}")
print(f"  genes           {final_df['Gene Symbol'].nunique()}")
for column in ("BEhive_CBE", "BEhive_ABE", "Hsu2013"):
    print(f"  {column:<15} {final_df[column].notna().sum():,} scored")
final_df.head()

## 10. Verification

Structural checks on the assembled library, followed by a comparison against the published design.

In [ ]:
structural = lib.structural_checks(final_df)
print(structural.to_string(index=False))
assert structural["passed"].all(), "structural invariants failed"

comparison = lib.published_comparison(final_df, stats_df)
print()
print(comparison.to_string(index=False))

if comparison["matches"].all():
    print("All checks passed. This run reproduces the published library.")
elif designs["regenerated"]:
    print("Designs were regenerated against a live Ensembl release, so differences")
    print("from the published library are expected. Guides that are new or changed")
    print("have no corresponding BEhive or FlashFry score, which is why the score")
    print("counts differ. The structural checks above still hold.")
else:
    raise AssertionError(
        "Run on the supplied inputs but produced different numbers: %s"
        % ", ".join(comparison.loc[~comparison["matches"], "check"]))

## 11. Run summary

Everything written by this notebook.

In [ ]:
written = sorted(p for p in OUT.rglob("*") if p.is_file())
sizes = pd.Series([p.stat().st_size for p in written],
                  index=[p.relative_to(OUT).as_posix() for p in written])

print(f"{len(written)} files written to {OUT}\n")
print(sizes.rename("bytes").to_string())